# **Cycle 4 Independent Notebook**

### *Isabella Bartoletti*

## 1. Environment Setup

In [116]:
%pip install pandas numpy

from pathlib import Path
import json
import pandas as pd
import numpy as np
import requests

DATA_DIR = Path("./")
FIXTURE_DIR = DATA_DIR / "api-fixtures"

# checks is api-fixtures folder exists
assert FIXTURE_DIR.exists(), "Run from the teaching-notebook directory and keep datasets beside it."

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


## 2. API Documentation

#### For this Notebook, I'm using PokeAPI: https://pokeapi.co. Specifically, I will use the `battle-armor` (from Pokémon Abilities endpoint) and `aegislash` (from Pokémon Species endpoint).

#### According to PokeAPI documentation, no API key, authentication, or parameters are required.



#### **Additionally, I will be working with the followings fields:**         
#### From `battle_armor`, I'm interested in investigating: 
- The list of Pokémon (`pokemon`) → dtype: `NamedAPIResource`
- Ability's `slot` number → dtype: `int`
- Ability's `is_hidden` status → dtype: `bool`
- `effect_entries`, aka what ability does → dtype: list `VerboseEffect` (yes, this is an actually dtype that lists the effect description in a specific language)

#### From `aegislash`, I'd like to investigate:
- `capture_rate`, measured from 1-255. The higher the number, the easier the species is to catch → dtype: `int`
- `base_happiness`, measured from 1-255. The higher the number, the happier the  Pokémon! → dtype: `int`
- `gender_rate`, or the chance of catching a female Pokémon measured in eighths (ex. 1 = 12.5% likelihood of catching a female Pokémon); or -1 for genderless Pokémon → dtype: `int`

#### *Note: I researched these field definitions on the PokeAPI documentation.*

#### **Loads JSON Pokémon fixtures:**

In [117]:
# saved copies
ability_path = FIXTURE_DIR / "battle-armor.json"
species_path = FIXTURE_DIR / "aegislash.json"

with ability_path.open(encoding="utf-8") as handle:
    ability = json.load(handle) # parses JSON contents in dict, then assigns it to fixture variable

with species_path.open(encoding="utf-8") as handle:
    species = json.load(handle)

# prints top level keys
print(ability.keys())
print(species.keys())

dict_keys(['id', 'name', 'is_main_series', 'generation', 'names', 'effect_entries', 'effect_changes', 'flavor_text_entries', 'pokemon'])
dict_keys(['id', 'name', 'order', 'gender_rate', 'capture_rate', 'base_happiness', 'is_baby', 'is_legendary', 'is_mythical', 'hatch_counter', 'has_gender_differences', 'forms_switchable', 'growth_rate', 'pokedex_numbers', 'egg_groups', 'color', 'shape', 'evolves_from_species', 'evolution_chain', 'habitat', 'generation', 'names', 'pal_park_encounters', 'form_descriptions', 'flavor_text_entries', 'genera', 'varieties'])


#### **Endpoints to investigate:**

In [118]:
ENDPOINT_ABILITY = "https://pokeapi.co/api/v2/ability/battle-armor"
ENDPOINT_SPECIES = "https://pokeapi.co/api/v2/pokemon-species/aegislash"

# Note that no parameters are needed for these endpoints

# Keys each response must contain (most of the fields I previously listed, and more!):
EXPECTED_ABILITY = {"id", "name", "effect_entries", "pokemon"} # battle armor abilities
EXPECTED_SPECIES = {"id", "name", "generation", "color", "shape", "habitat", "gender_rate", "capture_rate", "base_happiness", "is_legendary"}
EXPECTED_POKEMON_COUNT = 11 # got this number from referencing PokeAPI

## 3. Response Validation/Data Acquisition

#### **Payload Validation:**

In [119]:
# validate ability/battle-armor endpoint
def validate_ability_payload(payload): 
    if not isinstance(payload, dict):
        raise TypeError("API payload must be a JSON object") # ensures JSON object must be dict
    missing = EXPECTED_ABILITY - set(payload) # set difference: expected keys - keys actually found = missing keys (we don't want this)
    if missing: # False if missing empty set, True if missing non-empty
        raise ValueError(f"Missing ability fields: {sorted(missing)}")
    if len(payload["pokemon"]) != EXPECTED_POKEMON_COUNT: # number of pokemon with battle armor in my fixture
        raise ValueError(f"Expected {EXPECTED_POKEMON_COUNT} Pokémon, got {len(payload['pokemon'])}")
    return payload # only reached if every checked passed!
        

    # NOTE: I choose not to recycle the code below because my lists did not have to have the same length, unlike in EXPECTED_DAILY (one value per date)
    # lengths = {key: len(payload["daily"][key]) for key in EXPECTED_DAILY} # loop through keys in dict, value of each key number of values in the list
    #     if len(set(lengths.values())) != 1: # values in dict
    #        raise ValueError(f"Daily arrays have different lengths: {lengths}") # checking if array in the payload has the same length
    #    return payload

# validation species/aegislash (similar to above function)
def validate_species_payload(payload): 
    if not isinstance(payload, dict):
        raise TypeError("API payload must be a JSON object")
    missing = EXPECTED_SPECIES - set(payload) # missing keys
    if missing:
        raise ValueError(f"Missing species fields: {sorted(missing)}")
    return payload


# if not error occurs, fixtures passed validation
validate_species_payload(species)
validate_ability_payload(ability)

{'id': 4,
 'name': 'battle-armor',
 'is_main_series': True,
 'generation': {'name': 'generation-iii',
  'url': 'https://pokeapi.co/api/v2/generation/3/'},
 'names': [{'name': 'カブトアーマー',
   'language': {'name': 'ja-hrkt',
    'url': 'https://pokeapi.co/api/v2/language/1/'}},
  {'name': '전투무장',
   'language': {'name': 'ko', 'url': 'https://pokeapi.co/api/v2/language/3/'}},
  {'name': '戰鬥盔甲',
   'language': {'name': 'zh-hant',
    'url': 'https://pokeapi.co/api/v2/language/4/'}},
  {'name': 'Armurbaston',
   'language': {'name': 'fr', 'url': 'https://pokeapi.co/api/v2/language/5/'}},
  {'name': 'Kampfpanzer',
   'language': {'name': 'de', 'url': 'https://pokeapi.co/api/v2/language/6/'}},
  {'name': 'Armadura Batalla',
   'language': {'name': 'es', 'url': 'https://pokeapi.co/api/v2/language/7/'}},
  {'name': 'Lottascudo',
   'language': {'name': 'it', 'url': 'https://pokeapi.co/api/v2/language/8/'}},
  {'name': 'Battle Armor',
   'language': {'name': 'en', 'url': 'https://pokeapi.co/api/v2

#### **Data Acquisition:**

##### So that I don't have to do more DRY coding, I'm going to try making a generalized `acquire` function.

**The purpose of timeout:** the longest my code will wait for server to answer; without it, a slow server could make the notebook hang forever. In this case, after 10 seconds `requests` raises a timeout error, and my `except` block catches it and switches to the fixture (saved copy).
      
The HTTP status codes, on the other hand, are numbers that the server sends back to confirm whether GET request worked. `raise_for_status` checks this in code.
   
So far, we've learned a few popular codes, like `200` (OK, successful request), `404` (not found), and even more impractical ones (ex. `418` I'm a teapot).

In [ ]:
# Generalized data acquire function
def acquire(endpoint, fixture, validator, use_live=False): # user_live=False --> default parameter set to offline
    source = "cached fixture"
    payload = fixture # setting payload = fixture (saved copy), and if use_live is False then it just uses the fixture
    if use_live:
        try: 
            response = requests.get(endpoint, timeout=10) # sending a GET request using the request library, PokeAPI has no Header parameters
            response.raise_for_status() # checks the HTTP status code; turns HTTP errors (like 404) into exceptions 
            payload = response.json() # converts the response to JSON
            validator(payload)
            source = response.url # URl of the source
        except (requests.RequestException, ValueError, TypeError) as exc:
            print(f"Live request unavailable ({type(exc).__name__}); using fixture.")
            payload = fixture # live request unavailable, fall back to fixture
    return validator(payload), source

# made use_live = True so I can access live data, but if you wanted, you could change to use_live = False
ability_payload, ability_source = acquire(ENDPOINT_ABILITY, ability, validate_ability_payload, use_live = True)
species_payload, species_source = acquire(ENDPOINT_SPECIES, species, validate_species_payload, use_live = True)
print(ability_source)
print(species_source)

https://pokeapi.co/api/v2/ability/battle-armor
https://pokeapi.co/api/v2/pokemon-species/aegislash


## 4. Normalize into Dataframes

#### **Note: this normalization will be slightly more complicated than the original weather normalize code because the PokeAPI data has a different shape.**

Source for `json_normalize()`: https://pandas.pydata.org/docs/reference/api/pandas.json_normalize.html
     
And `rename()`: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rename.html

In [121]:
armor_df = pd.json_normalize(ability["pokemon"]) # transforms nested JSON dict into flat table (nested keys become "pokemon.name", "pokemon.url")
armor_df = armor_df.rename(columns={"pokemon.name": "pokemon_name"}) # clearer column name
armor_df = armor_df[["pokemon_name", "is_hidden", "slot"]] # columns i'm interested in
assert len(armor_df) == len(ability["pokemon"]) # check one row per Pokemon
assert {"pokemon_name", "is_hidden", "slot"} <= set(armor_df.columns) #check expected columns exist
armor_df

,pokemon_name,is_hidden,slot
0,cubone,True,3
1,marowak,True,3
2,kabuto,False,2
3,kabutops,False,2
4,anorith,False,1
5,armaldo,False,1
6,skorupi,False,1
7,drapion,False,1
8,type-null,False,1
9,perrserker,False,1


In [122]:
species_df = pd.json_normalize(species)
species_df = species_df[["id", "name", "generation.name", "color.name", "shape.name", "habitat", "gender_rate", "capture_rate", "base_happiness", "is_legendary"]] # note that this "." notation comes from doing json_normalize()
species_df = species_df.rename(columns={"id": "species_id", "generation.name": "generation", "color.name": "color", "shape.name": "shape"}) # streamlining naming, getting rid of "."
armor_df = armor_df[["pokemon_name", "is_hidden", "slot"]] # columns i'm interested in
assert len(species_df) == 1 # we only investigating one species, aegishlash
assert {"species_id", "capture_rate", "base_happiness", "gender_rate"} <= set(species_df.columns)
species_df

,species_id,name,generation,color,shape,habitat,gender_rate,capture_rate,base_happiness,is_legendary
0,681,aegislash,generation-vi,brown,blob,None,4,45,70,False


#### **Now, for the Complete Data Overview:**

In [ ]:
print("shapes:", armor_df.shape, species_df.shape) # (rows, columns)
print(armor_df.dtypes) # data types
print(species_df.dtypes)
print("missing values:", armor_df.isna().sum().sum(), species_df.isna().sum().to_dict()) # missing values (taking sum of missing values for all the keys)
print("duplicate rows:", armor_df.duplicated().sum()) 

shapes: (11, 3) (1, 10)
pokemon_name    object
is_hidden         bool
slot             int64
dtype: object
species_id         int64
name              object
generation        object
color             object
shape             object
habitat           object
gender_rate        int64
capture_rate       int64
base_happiness     int64
is_legendary        bool
dtype: object
missing values: 0 {'species_id': 0, 'name': 0, 'generation': 0, 'color': 0, 'shape': 0, 'habitat': 1, 'gender_rate': 0, 'capture_rate': 0, 'base_happiness': 0, 'is_legendary': 0}
duplicate rows: 0


##### The above check shows that `armor_df` has 11 rows (one per Pokemon), no missing values, and no duplicates. `species_df` has one row, but `habitat` is missing.

#### **Provenance formatting:**

In [124]:
provenance = {
    "provider": "PokeAPI",
    "endpoint": [ENDPOINT_ABILITY, ENDPOINT_SPECIES],
    "acquisition_source": {"battle_armor" : ability_source, "aegislash" : species_source},
    "fixture_retrieved": "2026-11-06",
    "attribution": "Data from PokeAPI: https://pokeapi.co",
}
provenance

{'provider': 'PokeAPI',
 'endpoint': ['https://pokeapi.co/api/v2/ability/battle-armor',
  'https://pokeapi.co/api/v2/pokemon-species/aegislash'],
 'acquisition_source': {'battle_armor': 'https://pokeapi.co/api/v2/ability/battle-armor',
  'aegislash': 'https://pokeapi.co/api/v2/pokemon-species/aegislash'},
 'fixture_retrieved': '2026-11-06',
 'attribution': 'Data from PokeAPI: https://pokeapi.co'}

# **See my Reflections Questions in the README.md File for *further elaboration*!!**